In [6]:
import pandas as pd
import numpy as np
import glob
import gc
import os

In [7]:
chunk_files = sorted(glob.glob("../data/interim/master_dataset_parquet/*.parquet"))

print(f"Number of chunks: {len(chunk_files)}")
print(chunk_files)

Number of chunks: 11
['master_dataset_parquet\\chunk_0001.parquet', 'master_dataset_parquet\\chunk_0002.parquet', 'master_dataset_parquet\\chunk_0003.parquet', 'master_dataset_parquet\\chunk_0004.parquet', 'master_dataset_parquet\\chunk_0005.parquet', 'master_dataset_parquet\\chunk_0006.parquet', 'master_dataset_parquet\\chunk_0007.parquet', 'master_dataset_parquet\\chunk_0008.parquet', 'master_dataset_parquet\\chunk_0009.parquet', 'master_dataset_parquet\\chunk_0010.parquet', 'master_dataset_parquet\\chunk_0011.parquet']


In [ ]:
df = pd.read_parquet(chunk_files[0])


In [9]:
print(df.shape)

df.head()

(5739000, 23)


,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,...,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,None,None,None,None,0,0,0,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,None,None,None,None,0,0,0,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,None,None,None,None,0,0,0,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,None,None,None,None,0,0,0,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,None,None,None,None,0,0,0,NaN


In [10]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5739000 entries, 0 to 5738999
Data columns (total 23 columns):
 #   Column        Dtype  
---  ------        -----  
 0   id            object 
 1   item_id       object 
 2   dept_id       object 
 3   cat_id        object 
 4   store_id      object 
 5   state_id      object 
 6   day           object 
 7   sales         int64  
 8   date          object 
 9   wm_yr_wk      int64  
 10  weekday       object 
 11  wday          int64  
 12  month         int64  
 13  year          int64  
 14  d             object 
 15  event_name_1  object 
 16  event_type_1  object 
 17  event_name_2  object 
 18  event_type_2  object 
 19  snap_CA       int64  
 20  snap_TX       int64  
 21  snap_WI       int64  
 22  sell_price    float64
dtypes: float64(1), int64(8), object(14)
memory usage: 4.0 GB


In [13]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,5739000,3000,HOBBIES_1_001_CA_1_validation,1913,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item_id,5739000,3000,HOBBIES_1_001,1913,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dept_id,5739000,7,FOODS_3,1480662,NaN,NaN,NaN,NaN,NaN,NaN,NaN
cat_id,5739000,3,FOODS,2655244,NaN,NaN,NaN,NaN,NaN,NaN,NaN
store_id,5739000,1,CA_1,5739000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state_id,5739000,1,CA,5739000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day,5739000,1913,d_1,3000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sales,5739000.0,NaN,NaN,NaN,1.300489,3.996753,0.0,0.0,0.0,1.0,599.0
date,5739000,1913,2011-01-29,3000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wm_yr_wk,5739000.0,NaN,NaN,NaN,11339.191323,150.374194,11101.0,11217.0,11333.0,11448.0,11613.0


In [12]:
missing = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": (df.isnull().mean() * 100).round(2)
})

missing = missing[missing["Missing Values"] > 0]

missing.sort_values("Percentage", ascending=False)

,Missing Values,Percentage
event_name_2,5727000,99.79
event_type_2,5727000,99.79
event_name_1,5277000,91.95
event_type_1,5277000,91.95
sell_price,1111614,19.37


In [14]:
(df["day"] == df["d"]).all()

True

In [15]:
df.drop(columns=["d"], inplace=True)

In [17]:
# Convert low-cardinality object columns to category
category_cols = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "weekday",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in category_cols:
    df[col] = df[col].astype("category")

In [18]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5739000 entries, 0 to 5738999
Data columns (total 22 columns):
 #   Column        Dtype   
---  ------        -----   
 0   id            category
 1   item_id       category
 2   dept_id       category
 3   cat_id        category
 4   store_id      category
 5   state_id      category
 6   day           object  
 7   sales         int64   
 8   date          object  
 9   wm_yr_wk      int64   
 10  weekday       category
 11  wday          int64   
 12  month         int64   
 13  year          int64   
 14  event_name_1  category
 15  event_type_1  category
 16  event_name_2  category
 17  event_type_2  category
 18  snap_CA       int64   
 19  snap_TX       int64   
 20  snap_WI       int64   
 21  sell_price    float64 
dtypes: category(11), float64(1), int64(8), object(2)
memory usage: 1.1 GB


In [19]:
before_memory = df.memory_usage(deep=True).sum() / 1024**2

print(f"Memory Before: {before_memory:.2f} MB")

Memory Before: 1086.52 MB


In [20]:
# Integer columns
df["sales"] = df["sales"].astype("int16")

df["wm_yr_wk"] = df["wm_yr_wk"].astype("int16")

df["wday"] = df["wday"].astype("int8")
df["month"] = df["month"].astype("int8")

df["year"] = df["year"].astype("int16")

df["snap_CA"] = df["snap_CA"].astype("int8")
df["snap_TX"] = df["snap_TX"].astype("int8")
df["snap_WI"] = df["snap_WI"].astype("int8")

# Float column
df["sell_price"] = df["sell_price"].astype("float32")

In [21]:
after_memory = df.memory_usage(deep=True).sum() / 1024**2

print(f"Memory After : {after_memory:.2f} MB")

reduction = (before_memory - after_memory) / before_memory * 100

print(f"Memory Reduction: {reduction:.2f}%")

Memory After : 774.55 MB
Memory Reduction: 28.71%


In [22]:
df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5739000 entries, 0 to 5738999
Data columns (total 22 columns):
 #   Column        Dtype   
---  ------        -----   
 0   id            category
 1   item_id       category
 2   dept_id       category
 3   cat_id        category
 4   store_id      category
 5   state_id      category
 6   day           object  
 7   sales         int16   
 8   date          object  
 9   wm_yr_wk      int16   
 10  weekday       category
 11  wday          int8    
 12  month         int8    
 13  year          int16   
 14  event_name_1  category
 15  event_type_1  category
 16  event_name_2  category
 17  event_type_2  category
 18  snap_CA       int8    
 19  snap_TX       int8    
 20  snap_WI       int8    
 21  sell_price    float32 
dtypes: category(11), float32(1), int16(3), int8(5), object(2)
memory usage: 774.5 MB


In [23]:
import os

os.makedirs("../data/interim/processed_chunks", exist_ok=True)

In [24]:


# جميع ملفات الـChunks
chunk_files = sorted(glob.glob("../data/interim/master_dataset_parquet/*.parquet"))

# مكان حفظ الملفات المعالجة
OUTPUT_DIR = "../data/interim/processed_chunks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for file in chunk_files:

    print(f"Processing {os.path.basename(file)}")

    # ==========================
    # Load
    # ==========================
    df = pd.read_parquet(file)

    # ==========================
    # Remove duplicate column
    # ==========================
    if "d" in df.columns:
        df.drop(columns=["d"], inplace=True)

    # ==========================
    # Category columns
    # ==========================
    category_cols = [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "weekday",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]

    for col in category_cols:
        df[col] = df[col].astype("category")

    # ==========================
    # Downcasting
    # ==========================
    df["sales"] = df["sales"].astype("int16")
    df["wm_yr_wk"] = df["wm_yr_wk"].astype("int16")

    df["wday"] = df["wday"].astype("int8")
    df["month"] = df["month"].astype("int8")

    df["year"] = df["year"].astype("int16")

    df["snap_CA"] = df["snap_CA"].astype("int8")
    df["snap_TX"] = df["snap_TX"].astype("int8")
    df["snap_WI"] = df["snap_WI"].astype("int8")

    df["sell_price"] = df["sell_price"].astype("float32")

    # ==========================
    # Save
    # ==========================
    output_file = os.path.join(
        OUTPUT_DIR,
        os.path.basename(file)
    )

    df.to_parquet(
        output_file,
        index=False
    )

    print(f"Saved -> {output_file}")

    del df
    gc.collect()

print("\nAll chunks processed successfully.")

Processing chunk_0001.parquet
Saved -> processed_chunks\chunk_0001.parquet
Processing chunk_0002.parquet
Saved -> processed_chunks\chunk_0002.parquet
Processing chunk_0003.parquet
Saved -> processed_chunks\chunk_0003.parquet
Processing chunk_0004.parquet
Saved -> processed_chunks\chunk_0004.parquet
Processing chunk_0005.parquet
Saved -> processed_chunks\chunk_0005.parquet
Processing chunk_0006.parquet
Saved -> processed_chunks\chunk_0006.parquet
Processing chunk_0007.parquet
Saved -> processed_chunks\chunk_0007.parquet
Processing chunk_0008.parquet
Saved -> processed_chunks\chunk_0008.parquet
Processing chunk_0009.parquet
Saved -> processed_chunks\chunk_0009.parquet
Processing chunk_0010.parquet
Saved -> processed_chunks\chunk_0010.parquet
Processing chunk_0011.parquet
Saved -> processed_chunks\chunk_0011.parquet

All chunks processed successfully.


In [25]:
df["sell_price"].isna().sum()

NameError: name 'df' is not defined

In [27]:
event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

for col in event_cols:

    df[col] = (
        df[col]
        .cat.add_categories(["No_Event"])
        .fillna("No_Event")
    )

In [28]:
df[event_cols].isna().sum()

event_name_1    0
event_type_1    0
event_name_2    0
event_type_2    0
dtype: int64

In [29]:
missing_price = df[df["sell_price"].isna()]

missing_price["sales"].value_counts().head(20)

sales
0    12299413
Name: count, dtype: int64

In [30]:
missing_price["sales"].describe()

count    12299413.0
mean            0.0
std             0.0
min             0.0
25%             0.0
50%             0.0
75%             0.0
max             0.0
Name: sales, dtype: float64

In [31]:
df.isnull().sum()[df.isnull().sum() > 0]

sell_price    12299413
dtype: int64

In [32]:
def preprocess_chunk(df):

    # Drop duplicate column
    df.drop(columns=["d"], inplace=True)

    # Category conversion
    category_cols = [
        "id", "item_id", "dept_id", "cat_id",
        "store_id", "state_id", "weekday",
        "event_name_1", "event_type_1",
        "event_name_2", "event_type_2"
    ]

    for col in category_cols:
        df[col] = df[col].astype("category")

    # Numeric downcasting
    df["sales"] = df["sales"].astype("int16")
    df["wm_yr_wk"] = df["wm_yr_wk"].astype("int16")
    df["wday"] = df["wday"].astype("int8")
    df["month"] = df["month"].astype("int8")
    df["year"] = df["year"].astype("int16")
    df["snap_CA"] = df["snap_CA"].astype("int8")
    df["snap_TX"] = df["snap_TX"].astype("int8")
    df["snap_WI"] = df["snap_WI"].astype("int8")
    df["sell_price"] = df["sell_price"].astype("float32")

    # Event columns
    event_cols = [
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]

    for col in event_cols:
        df[col] = (
            df[col]
            .cat.add_categories(["No_Event"])
            .fillna("No_Event")
        )

    return df

In [34]:
INPUT_DIR = "../data/interim/master_dataset_parquet"
OUTPUT_DIR = "../data/interim/preprocessed_chunks"

os.makedirs(OUTPUT_DIR, exist_ok=True)


def preprocess_chunk(df):

    # =========================
    # Remove duplicate column
    # =========================
    if "d" in df.columns:
        df.drop(columns=["d"], inplace=True)

    # =========================
    # Category columns
    # =========================
    category_cols = [
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id",
        "weekday",
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]

    for col in category_cols:
        df[col] = df[col].astype("category")

    # =========================
    # Numeric Downcasting
    # =========================
    df["sales"] = df["sales"].astype("int16")
    df["wm_yr_wk"] = df["wm_yr_wk"].astype("int16")

    df["wday"] = df["wday"].astype("int8")
    df["month"] = df["month"].astype("int8")

    df["year"] = df["year"].astype("int16")

    df["snap_CA"] = df["snap_CA"].astype("int8")
    df["snap_TX"] = df["snap_TX"].astype("int8")
    df["snap_WI"] = df["snap_WI"].astype("int8")

    df["sell_price"] = df["sell_price"].astype("float32")

    # =========================
    # Event Missing Values
    # =========================
    event_cols = [
        "event_name_1",
        "event_type_1",
        "event_name_2",
        "event_type_2"
    ]

    for col in event_cols:

        if "No_Event" not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories(["No_Event"])

        df[col] = df[col].fillna("No_Event")

    return df


chunk_files = sorted(glob.glob(f"{INPUT_DIR}/*.parquet"))

print(f"Found {len(chunk_files)} chunks")

for file in chunk_files:

    print(f"\nProcessing {os.path.basename(file)}")

    df = pd.read_parquet(file)

    df = preprocess_chunk(df)

    output_path = os.path.join(
        OUTPUT_DIR,
        os.path.basename(file)
    )

    df.to_parquet(
        output_path,
        index=False
    )

    print(
        f"Saved -> {output_path} | "
        f"Memory: {df.memory_usage(deep=True).sum()/1024**2:.2f} MB"
    )

    del df
    gc.collect()

print("\nDay 4 preprocessing completed successfully.")

Found 11 chunks

Processing chunk_0001.parquet
Saved -> preprocessed_chunks\chunk_0001.parquet | Memory: 774.55 MB

Processing chunk_0002.parquet
Saved -> preprocessed_chunks\chunk_0002.parquet | Memory: 774.55 MB

Processing chunk_0003.parquet
Saved -> preprocessed_chunks\chunk_0003.parquet | Memory: 774.55 MB

Processing chunk_0004.parquet
Saved -> preprocessed_chunks\chunk_0004.parquet | Memory: 774.55 MB

Processing chunk_0005.parquet
Saved -> preprocessed_chunks\chunk_0005.parquet | Memory: 774.55 MB

Processing chunk_0006.parquet
Saved -> preprocessed_chunks\chunk_0006.parquet | Memory: 774.55 MB

Processing chunk_0007.parquet
Saved -> preprocessed_chunks\chunk_0007.parquet | Memory: 774.55 MB

Processing chunk_0008.parquet
Saved -> preprocessed_chunks\chunk_0008.parquet | Memory: 774.55 MB

Processing chunk_0009.parquet
Saved -> preprocessed_chunks\chunk_0009.parquet | Memory: 774.55 MB

Processing chunk_0010.parquet
Saved -> preprocessed_chunks\chunk_0010.parquet | Memory: 774.

NameError: name 'df' is not defined